# 自己回帰型ニューラルネットワークと時系列処理

PyTorchによる自己回帰型ニューラルネットワーク（Recurrent Neural Network：RNN）の記述方法を学習します．  
今回は長短期記憶（Long Short Term Memory：LSTM）を持ったRNNの記述方法を学習します．  

**目標：RNNとMLPの違いを理解**

---

例題では，回帰問題（気温予測）を解きます．  
演習では，分類問題（季節分類）を解きます．

※rnn_data.csvというデータセットを使います  
　2015〜2020年の日中平均気温と日付が記録されたデータセットです

---
## この教材について

「3分で学ぶPyTorch」シリーズの **RNN（自己回帰型ニューラルネットワーク） 基礎編（第2回）** の演習パートです。

このノートブックは**演習ファイル（task）**です。`【TASK】` と書かれた箇所を埋めて実行してください。詰まったときは同じ回の回答ファイル（ans）を参照してください。

この教材と関連記事は note で無料公開しています。
シリーズ一覧: https://note.com/technosend/m/m84d841b6d067

---

## rnn_data.csvデータセット
- 2015〜2020年の日中平均気温と日付が記録されたデータセット
  - 第1カラム：日付
  - 第2カラム：日中の平均気温
- 全部で2192個
  365*6+2（2016年と2020年が閏年なため）
- 学習データ：先頭1826個（2016年から2019年までのデータ）
- テストデータ：後ろから366個（2020年のデータ）
- 値域：-2.0~32.8  
  最低平均気温が-2℃で最高平均気温が32.8℃
- 例題：気温予測での使い方  
  - 50日分のデータを1つの時系列として学習させる  
  - 今の日付に対して1日後を教師データとする  
  （ = 入力データ1つにつき，1つの教師データがある）
  - データの形状は50, 1776, 1になる  
  （ = 50日で1セットのデータが1776個分ある）
  - テスト時には365日分のデータを1セットにして入力して出力させる  
  ※本来であれば50日を1セットにした方がいいが，プログラムの簡単化のためこの仕様にしている
- 演習：季節分類での使い方
  - 50日分のデータを1つの時系列として学習させる 
  - 50日分のデータに対して1つの季節分類を教師データとする  
  （ =49日分の出力は無視して50日目の出力を分類結果とする）
    - 季節分類は日付のデータを次の規則で変換させる
      - 3〜5月：春
      - 6〜8月：夏
      - 9〜11月：秋
      - 12〜2月：冬
  - データの形状は50, 1776, 1になる  
  （ = 50日で1セットのデータが1776個分ある）
  - テスト時には365日分のデータを1セットにして入力して出力させる  
  ※本来であれば50日で1セットにして49日分無視して50日目のデータを取得する方がいいが，  
  出力の見栄えとプログラムの簡単化のためこの仕様にしている


## 演習 RNNによる分類問題

**LSTM2層・全結合1層のRNNの作成**  
平均気温を時系列データとして順次入力し，季節を出力する分類問題を解く．



### 演習1. ライブラリのインポート


深層学習演算ライブラリPyTorchなどのライブラリ，パッケージ，モジュールをインポートする．  

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=14XdT7XWs6JzTil6JhZZw7aM3VpAxdYH2&sz=w400">


#### 演習1のコード

In [ ]:
# 演習1. ライブラリのインポート

import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt
# japanize-matplotlib は古いパッケージです。
# 代替: !pip install matplotlib-fontja -q  (2023年以降推奨)
!pip install japanize-matplotlib -q
import japanize_matplotlib

import re
!pip install gdown --upgrade -q  # gdown 4.6以降はアップグレード推奨
import gdown


#### 今回使うパッケージ，モジュール一覧



- torch：多次元テンソルのデータ構造とそのテンソルのための算術演算が組み込まれたパッケージ
- torch.nn：ニューラルネットワークを定義するためのパッケージ  
nnという略称を与えることが多い
- torch.optim：最適化器を宣言するためのパッケージ  
optimという略称を与えることが多い
- numpy：行列演算を行うライブラリ（今回は画像の表示のために使用）  
npという略称を与えることが多い
- matplotlib.pyplot：グラフや画像の描画を行うモジュール  
pltという略称を与えることが多い
- japanize_matplotlib：Matplotlibで日本語を表示するためのライブラリ

- [re](https://docs.python.org/ja/3/library/re.html)：正規表現操作を行うライブラリ
- gdown：GoogleDriveからファイルをダウンロードするためのライブラリ


<font color="blue">【TASK】</font>パッケージをインポートしましょう

### 演習2. ニューラルネットワークの定義

ニューラルネットワーククラスを定義して，そのクラスのインスタンスを宣言する．

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1jl1W_RW7HrU0ksk1a0XrSq6CyldXF4qZ&sz=w400">


#### 演習2のコード

In [ ]:
# 演習2. ニューラルネットワークの定義

# 1. ニューラルネットワーククラスの定義
class TempClassifier(nn.Module):
    def __init__(self):
        super(TempClassifier, self).__init__()
        # 【TASK】LSTM層と全結合層を宣言
        
    def forward(self, x):
        # 【TASK】順伝播のパスを定義
        return x
 
# 2. インスタンスの宣言
temp_classifier = # 【TASK】ニューラルネットワーククラスのインスタンスの宣言
device = # 【TASK】GPUの指定
# 【TASK】GPUにセットアップ

#### 1. ニューラルネットワーククラスの定義  
  - nn.Moduleを継承したクラス「TempClassifier」を定義
  - \_\_init\_\_()とforward()を定義

    ```python
    # 1.ニューラルネットワーククラスの定義
    class TempClassifier(nn.Module):
        def __init__(self):
            super(TempClassifier, self).__init__()
            # 【TASK】LSTM層と全結合層を宣言
        def forward(self, x):
            # 【TASK】順伝播のパスを定義
            return x
    ```


<font color="blue">【TASK】</font>LSTM層と全結合層をnn.LSTMクラスとnn.Linearクラスを用いて宣言しましょう  
ニューラルネットワークの構成は下記の通りです  
- LSTM層１：入力1，出力10，レイヤー数2
- 全結合層：入力10，出力4  
※レイヤー数を2にした時のNNの様子は[こちら](https://drive.google.com/file/d/1NGLeTjsp6i2c4ltLpy2NbQVNTSyq0Hyl/view?usp=sharing)


<font color="blue">【TASK】</font>順伝播のパスを定義しましょう  
ある時刻のデータ（数値）を入力すると想定し，順伝播のパスの構成は下記の通りです  
1. LSTM層
2. 全結合層  

※PyTorchのLSTM層は内部に活性化関数を持つので，今回は2つの層の間に活性化関数がありません．


#### 2. インスタンスの宣言

- temp_classifierという名前でインスタンスを宣言
    ```python
    temp_classifier = # 【TASK】ニューラルネットワーククラスのインスタンスの宣言
    ```

- GPUにセットアップ

    ```python
    device = # 【TASK】GPUの指定
    # 【TASK】GPUにセットアップ
    ```

- ニューラルネットワーククラスのインスタンスの宣言

  <font color="blue">【TASK】</font>ニューラルネットワーククラスのインスタンスを宣言しましょう 

- GPUにセットアップ
  - Moduleクラスにもto()がある
  - Tensor型変数同様にto()を使ってGPUにデータを渡すことができる
  - 学習時にGPUを使う場合は，ニューラルネットワーククラスのインスタンスをGPUに渡す

  <font color="blue">【TASK】</font>ニューラルネットワーククラスのインスタンスをGPUにセットアップしましょう  
  - デバイス指定用の変数deviceを宣言しましょう
  - `torch.cuda.is_available()` でGPU有無を判定し，利用可能な場合は"cuda:0"，利用不可の場合は"cpu"を指定しましょう
    - torch.deviceクラスを使ってデバイスを指定しましょう
  - to()を使ってGPUにセットアップしましょう


### 演習3. 誤差関数・最適化器の設定

誤差関数と最適化器を宣言する．

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1oS8_oitSvcQ9f5_uYMrDXyC3P_vrM7jp&sz=w400">


#### 演習3のコード

In [ ]:
# 演習3. 誤差関数・最適化器の設定

# 1. 誤差関数の宣言
criterion = # 【TASK】誤差関数の宣言

# 2. 最適化器の宣言
optimizer_temp_classifier = # 【TASK】最適化器の宣言

#### 1. 誤差関数の宣言

- クロスエントロピー誤差を計算するcriterionを宣言

    ```python
    criterion = # 【TASK】誤差関数の宣言
    ```


<font color="blue">【TASK】</font>誤差関数を宣言しましょう  
- クロスエントロピー誤差関数を使いましょう
- nn.CrossEntropyLossクラスを用いて宣言しましょう

#### 2. 最適化器の宣言

- Adamを計算するoptimizer_temp_classifierを宣言

    ```python
    optimizer_temp_classifier = # 【TASK】最適化器の宣言
    ```

<font color="blue">【TASK】</font>最適化器を宣言しましょう  
- Adamを使いましょう
- [optim.Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html#adam)クラスを用いて宣言しましょう  
- 引数の構成は下記の通りです  
  - ニューラルネットワークのパラメータtemp_classifierのパラメータ

### 演習4. データセットの作成

自作した関数make_seasonal_dataset()を使ってデータセットを作成する．

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=17fS-oMI83rSL7SxN_GyKHjxP-FO-R3aQ&sz=w400">


#### 演習4のコード

まずはgdown.download()を使ってファイルをダウンロードします．
```python
gdown.download(id=file_id, output="rnn_data.csv", quiet=True)
# id：Google DriveのファイルID(str)  ※ gdown 4.6以降の推奨形式
# output（オプション）：ダウンロード時のファイル名(str)
# quiet（オプション）：ダウンロードのログを出力するかどうか(bool)
```

<font color="blue">【TASK】</font>セルを実行してファイルをダウンロードしましょう

In [ ]:
# データのダウンロード
# 旧形式URL変換済み: uc?export=download は gdown 4.6以降で不安定
file_id = "1ShNdssPwqjcMXgI_z_HDNI7ei6oLimGu"  # Google Drive ファイルID
gdown.download(id=file_id, output="rnn_data.csv", quiet=True)

In [ ]:
# 演習4. データセットの準備
# 1. データセットを作成する関数の用意

# 日付を気象学的季節へ変換
def date_to_season(date):
    # 日付に対応する季節のlistを作成
    season = []
    for d in date:
      s = re.findall('/(.*)/', str(d))
      # 春
      if s == ['3'] or s == ['4'] or s == ['5']:
        season.append(0)
      # 夏
      elif s == ['6'] or s == ['7'] or s == ['8']:
        season.append(1)
      # 秋
      elif s == ['9'] or s == ['10'] or s == ['11']:
        season.append(2)
      # 冬
      else:
        season.append(3)
    return season

# 気温データと日付データからデータセットを作成する関数
def make_seasonal_dataset(data, date, train=True, seq_len=50):

    # 日付から季節に変換
    season = date_to_season(date)

    # train=True：学習データの作成
    if train:
        # データセットを格納するlistの初期化
        inputs = []
        labels = []
        
        # 指定した系列長だけ取得
        for t in range(seq_len):
            data_batch = []
            label_batch = []
            for i in range(len(data) - seq_len):
                data_batch.append(data[i+t : i+t+1])
                label_batch.append(season[i+t+1])
            inputs.append(data_batch)
            labels.append(label_batch)
    
        # numpyへ変換
        inputs = np.array(inputs, dtype=np.float32) 
        labels = np.array(labels, dtype=np.int64)

        # torchへ変換
        inputs = torch.tensor(inputs) 
        labels = torch.tensor(labels) 

    # train=False：テストデータの作成
    else:
        # numpyへ変換
        inputs = np.array(data, dtype=np.float32) 
        labels = np.array(season, dtype=np.int64)

        # torchへ変換
        inputs = torch.tensor(inputs).view(inputs.shape[0], 1, 1) 
        labels = torch.tensor(labels) 

    return [[inputs, labels]]

# 2. データの読み込み
# 気温データの列のみ読み込む
temp_data = np.loadtxt('./rnn_data.csv', delimiter=',', dtype=np.float32, usecols=1)
# 日付を文字列で取得
date = np.loadtxt('./rnn_data.csv', delimiter=',', dtype=np.unicode, usecols=0)

# 3. 値域を0〜1の範囲に正規化
temp_data = temp_data / 40

# 4. 気温データ，日付を学習データ，テストデータに分割
train_temp_data = temp_data[:-366] 
train_date = date[:-366] 
test_temp_data = temp_data[-366:]
test_date = date[-366:]

# 日時の長さを表示
print('学習データの日数：', len(train_temp_data))
print('テストデータの日数：', len(test_temp_data))

# 5. データセットの作成
seq_len = 50 
train_loader = make_seasonal_dataset(train_temp_data, train_date, train=True, seq_len=seq_len)
test_loader = make_seasonal_dataset(test_temp_data, test_date, train=False)
print("データセットの概要")
print("学習データ")
print(train_loader[0][0].shape)
print("テストデータ")
print(test_loader[0][0].shape)

#### 1. データセットを作成する関数の用意
  
  - date_to_season()：日付を気象学的季節へ変換する関数
  ```python
  date_to_season(date)
  # 第1引数：日付データ（strのlistかnumpy配列）
  # 戻り値1：季節データ(list)
  ```
  - make_seasonal_dataset()：気温データと予測する季節のラベルをデータセット化する関数
  ```python
  make_seasonal_dataset(data, date, train=True, seq_len=50)
  # 第1引数：気温データ
  # 第2引数：日付
  # 第3引数（オプション）：学習データを作成するか指定(bool)，デフォルトTrue
  # 第4引数（オプション）：時系列データの系列長(int)，デフォルト50
  # 戻り値1：入力データと教師データの配列(list)
  ```

  - train=Trueの場合，make_seasonal_dataset()では気温データをデータ長さに区切り，対応する日付を季節のラベルに変換して教師データとして用意
  - train=Falseの場合，気温データは特定のデータ長に区切らず，対応する日付を季節のラベルに変換して教師データとして用意
  - make_seasonal_dataset()内では日付を気象学的季節へ変換するdate_to_season()を呼び出す

#### 2. データの読み込み  

  - temp_dataという名前で気温データ用の変数を宣言
  - dateという名前で日付データ用の変数を宣言
  ```python
  temp_data = # 気温データの読み込み
  date = # 日付の読み込み
  ```
   

#### 3. 値域を0〜1の範囲に正規化

- 気温データの値域を正規化
- 正規化の結果はtemp_dataに格納

  ```python
  temp_data = # 正規化する
  ```

#### 4. 気温データと日付を学習データ，テストデータに分割

  - スライスにより気温データを学習データとテストデータに分割  
    - train_temp_dataという名前で学習用の気温データ用の変数を宣言  
    - train_dateという名前で学習に用いる日付データ用の変数を宣言  
    - test_temp_dataという名前でテスト用の気温データ用の変数を宣言  
    - test_dateという名前でテストに用いる日付データ用の変数を宣言  

  ```python
  train_temp_data = # 気温データをスライスし学習データを用意 
  train_date = # 日付データをスライスし学習に用いる日付データを用意 
  test_temp_data = # 気温データをスライスし学習データを用意
  test_date = # 日付データをスライスしテスト時の日付データを用意
  ```

#### 5. データセットの作成
  - seq_lenという名前でデータ長用の変数を宣言
  - train_data, train_labelという名前で学習データ用の変数を宣言
  - test_data, test_labelという名前でテストデータ用の変数を宣言
  
  ```python
  seq_len = # データ長を設定
  train_data, train_label = # 学習データの用意
  test_data, test_label = # テストデータの用意
  ```


- make_seasonal_dataset()を使ってデータセットを作成
  - nn.LSTMクラスでは入力するデータを[データ長 x バッチサイズ x 各時刻の入力サイズの3次元配列にする必要がある](https://drive.google.com/file/d/1LU8Yxb9tuaQf7MdJbVDLpuNZu2RNZ0U_/view?usp=sharing)
  - 50日分(seq_len = 50)を1つの時系列として扱う
  
  ```python
  train_loader = make_seasonal_dataset(train_temp_data, train_date, train=True, seq_len=seq_len)
  test_loader = make_seasonal_dataset(test_temp_data, test_date, train=False)
  ```


### 演習5. 学習

教師データとの誤差を計算し，パラメータを更新する．  

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1A6TjumoBejWpKGvD6TDQeivN1tpeEBp4&sz=w400">



#### 演習5のコード：前半の学習部分

In [ ]:
# 演習5. 学習

# 1. 学習ループの作成
epochs = # 【TASK】エポック数

# エポックのループ
for epoch in # 【TASK】エポック:
    # 学習データのデータローダーのループ
    for data in train_loader:
    
        # 2. ニューラルネットワークへのデータの入力
        inputs, labels = data
        inputs = # 【TASK】GPUにセットアップ
        labels = # 【TASK】GPUにセットアップ
        outputs = # 【TASK】ニューラルネットワークからの出力
        
        # 3. 誤差の計算（BPTTパターン2：最後の時刻に教師データがある場合)
        loss = # 【TASK】誤差計算

        # 4. 誤差逆伝播とパラメータの更新 
        # 【TASK】パラメータの微分値を初期化
        # 【TASK】誤差逆伝播
        # 【TASK】パラメータの更新
      
        # 現在の誤差の値の表示
        if (epoch + 1) % 100 == 0:
            print("epoch :", epoch + 1, "学習誤差 :",loss.item())

#### 1. 学習ループの作成  
- ミニバッチ学習を行う学習ループの作成
- epochsという名前でエポック用の変数を宣言
- 外側にエポック，内側に学習データのデータローダーのループを作成
```python
epochs = # 【TASK】エポック数
for epoch in # 【TASK】エポック
      for data in # 【TASK】学習データのデータローダー
```

<font color="blue">【TASK】</font>学習ループを作成しましょう  
ループの設定は下記の通りです
- エポック
  - エポック数1000
      - rangeクラスを使ってループさせましょう
  - 学習データのデータローダー
    - ループ対象：学習データのデータローダーtrain_loader
    - forを使ってtrain_loaderをループさせましょう

#### 2. ニューラルネットワークへのデータの入力  

- GPUにセットアップ
- outputsという名前で出力用の変数を宣言
```python
inputs, labels = data
inputs = # 【TASK】GPUにセットアップ
targets = # 【TASK】GPUにセットアップ
outputs = # 【TASK】ニューラルネットワークからの出力
```


  
<font color="blue">【TASK】</font>ニューラルネットワークへデータを入力しましょう
  - 入力inputsをGPUにセットアップしましょう
  - 教師データlabelsをGPUにセットアップしましょう
  - inputsをニューラルネットワークに与えて出力outputsを取得しましょう

#### 3. **<font color="red">【NEW!】</font>** 誤差の計算（BPTTパターン2：最後の時刻に教師データがある場合）

  - lossという名前で誤差計算の結果用の変数を宣言
  ```python
  loss = # 【TASK】誤差計算
  ```



- 誤差計算はnn.MSELossクラスやnn.CrossEntropyLossクラスなどの誤差関数クラスのインスタンスに引数を2つ与えて行う
```python
loss = criterion(outputs, labels)
# 第1引数：ニューラルネットワークの出力
# 第2引数：教師データ
```

<font color="blue">【TASK】</font>誤差を計算(最後の時刻に教師データがある場合）をしましょう  
設定は以下の通りです．
- 出力，教師データともに最後の時刻（シーケンスの最後）のデータ抜き出す
- 配列の最後のデータは-1を指定して抜き出す

#### 4. 誤差逆伝播とパラメータの更新  

- パラメータの微分値を初期化
- 誤差逆伝播
- パラメータの更新
```python
# 【TASK】パラメータの微分値を初期化
# 【TASK】誤差逆伝播
# 【TASK】パラメータの更新
```


- パラメータの微分値の初期化は最適化器が持つ
  zero_grad()を呼び出して行う
  
- 誤差逆伝播は計算結果を持った変数からbackward()を呼び出して行う
- パラメータの更新は最適化器の持つstep()を呼び出して行う  

<font color="blue">　【TASK】</font>誤差逆伝播とパラメータの更新を行いましょう
- zero_grad()を使ってパラメータの微分値の初期化をしましょう
- backward()を使って誤差逆伝播させましょう
- step()を使ってパラメータの更新を行いましょう





#### 演習5のコード：後半の予測部分

In [ ]:
# 5. テスト

# ニューラルネットワークへのデータ入力
inputs = # 【TASK】GPUにセットアップ
labels = # 【TASK】GPUにセットアップ
outputs = # 【TASK】ニューラルネットワークからの出力

# 予測ラベルの取得と正解率の算出
pred = # 【TASK】予測結果の取得
correct = # 【TASK】正解数の取得
print("test accuracy: {}".format(correct / len(pred)))

# 結果の表示
print('日付\t\t\t判定\t予測\t正解')
for i in range(len(pred)):
    if pred[i] == labels[i]:
        ans = 'OK'
    else:
        ans = 'NG'
    print(date[-366 + i], '\t', ans, '\t', pred[i].item(), '\t', labels[i].item())

#### 5. 季節分類のテスト
  - ニューラルネットワークへのデータ入力
  - 予測ラベルの取得と正解率の算出
  ```python
  inputs = # 【TASK】GPUにセットアップ
  labels = # 【TASK】GPUにセットアップ
  outputs = # 【TASK】ニューラルネットワークからの出力
  
  pred = # 【TASK】予測結果の取得
  correct = # 【TASK】正解数の取得
  ```

<font color="blue">【TASK】</font>季節分類のテストを行いましょう  
- テストデータと教師データをGPUにセットアップ
    - outputsという名前で出力用の変数を宣言
- 予測結果と教師データから正解数を取得しましょう
    - predという名前で予測結果用の変数を宣言  
    - correctという名前で正解数用の変数を宣言  

書き方がわからなければ[第2回例題5.5 予測精度の計算と表示](https://colab.research.google.com/drive/1KtH5SrTCeSUgLtSTncNbM-w5vflaqsht#scrollTo=8e5fvRIMRMof&forceEdit=true&sandboxMode=true)を参考にしてください  

## まとめ

今回は気温の時系列データを用いて，例題では気温予測問題，演習では季節分類問題に取り組みました．  
演習では高い分類性能が得られず，一見学習がうまくいっていないように見えますが，  
これは気温が年ごとに変動するため，そもそも気温から暦を推定することは困難だからです．  
ここでは季節の推定ではなく例年であれば◯月並みの気温だと予測するモデルができていたと考えられます．    
次の段階として自分でデータセットを用意して問題を設定することが挙げられます．  
併せてエポック数やNNの構造を変えたりすることで学習精度を向上させることができます．  
余力があればこれらの取り組みも行ってみてください．